In [ ]:
# ============================================================
# 🚀 FINAL PIPELINE (Version-Independent)
# NO evaluation_strategy, NO load_best_model_at_end
# Works on all old transformers versions
# ============================================================

!pip install -q --upgrade transformers accelerate datasets peft scikit-learn

import random
import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model


# -------------------------
# 1) Generate 600 headlines
# -------------------------
random.seed(42)

left_templates = [
    "Progressive leaders call for {policy} to support working families.",
    "Climate activists praise new {policy} as vital for the planet.",
    "Experts say {policy} will reduce inequality.",
]
right_templates = [
    "Business leaders warn {policy} will hurt economic growth.",
    "Critics say {policy} is government overreach.",
    "Conservatives oppose {policy}, calling it harmful regulation.",
]
neutral_templates = [
    "Government announces new {policy} after parliamentary discussions.",
    "Experts give mixed reactions to {policy}.",
    "Report outlines potential effects of {policy}.",
]

policy = [
    "healthcare reform","climate policy","tax legislation",
    "education funding","labor protections","immigration rules"
]

def generate(tpls,label,n):
    out=[]
    for _ in range(n):
        out.append({
            "headline":random.choice(tpls).format(policy=random.choice(policy)),
            "bias":label
        })
    return out

df = pd.DataFrame(
    generate(left_templates,"Left",200)+
    generate(right_templates,"Right",200)+
    generate(neutral_templates,"Neutral",200)
).sample(frac=1).reset_index(drop=True)

print("Dataset distribution:")
print(df["bias"].value_counts())

df.to_csv("train.csv", index=False)


# -------------------------
# 2) Prepare dataset
# -------------------------
LABELS = ["Left","Neutral","Right"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

df = df.rename(columns={"headline":"text","bias":"label"})
train_df,val_df = train_test_split(df,test_size=0.1,stratify=df["label"],random_state=42)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))


# -------------------------
# 3) Tokenize
# -------------------------
BASE = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE)

def tok(batch):
    return tokenizer(batch["text"],padding="max_length",truncation=True,max_length=128)

train_ds = train_ds.map(tok,batched=True)
val_ds   = val_ds.map(tok,batched=True)

train_ds = train_ds.rename_column("label","labels")
val_ds   = val_ds.rename_column("label","labels")

def encode(x):
    x["labels"]=label2id[x["labels"]]
    return x

train_ds=train_ds.map(encode)
val_ds  =val_ds.map(encode)

train_ds.set_format("torch",columns=["input_ids","attention_mask","labels"])
val_ds.set_format("torch",columns=["input_ids","attention_mask","labels"])


# -------------------------
# 4) Model + LoRA
# -------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    BASE,
    num_labels=len(LABELS),
    label2id=label2id,
    id2label=id2label
)

lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin","v_lin"],
    bias="none",
    task_type="SEQ_CLS"
)

peft = get_peft_model(model,lora)

# Make classifier trainable
for name,p in peft.named_parameters():
    if "classifier" in name or "pre_classifier" in name:
        p.requires_grad=True


# -------------------------
# 5) TrainingArguments (SAFE VERSION)
# -------------------------
training_args = TrainingArguments(
    output_dir="./bias_model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=8,
    weight_decay=0.01,
    learning_rate=4e-5,
    logging_steps=20,
    save_steps=500,          # Safe: supported everywhere
    eval_steps=500,
    report_to="none",        # Avoid wandb corruption
)

def metrics(pred):
    logits,labels=pred
    preds=np.argmax(logits,axis=1)
    return {
        "accuracy":accuracy_score(labels,preds),
        "f1":f1_score(labels,preds,average="macro")
    }

trainer = Trainer(
    model=peft,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=metrics
)

print("\n🚀 Training starting...\n")
trainer.train()


# -------------------------
# 6) EXPORT
# -------------------------
SAVE="/content/ft_lora_adapter"
os.makedirs(SAVE,exist_ok=True)

peft.save_pretrained(SAVE)

# Save classifier head
head_state={
    "pre_classifier":peft.base_model.pre_classifier.state_dict(),
    "classifier":peft.base_model.classifier.state_dict(),
}
torch.save(head_state,os.path.join(SAVE,"cls_head.pt"))

# labels
with open("le_classes.txt","w") as f: f.write(",".join(LABELS))

print("\n✅ Export complete.")
print("Saved: ft_lora_adapter/, cls_head.pt, le_classes.txt")

# Zip & Download
!zip -r ft_lora_adapter.zip ft_lora_adapter
from google.colab import files
files.download("ft_lora_adapter.zip")
files.download("le_classes.txt")

print("\n🎉 ALL DONE! Upload to GitHub next to app.py")


Dataset distribution:
bias
Left       200
Neutral    200
Right      200
Name: count, dtype: int64


Map:   0%|          | 0/540 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/540 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 ALL DONE! Upload to GitHub next to app.py


In [2]:
# ============================================================
# 0) INSTALL DEPENDENCIES
# ============================================================
!pip install -q --upgrade transformers accelerate datasets peft scikit-learn requests

import os, random, requests, math
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

# ============================================================
# 1) FETCH REAL NEWS HEADLINES FROM NEWSAPI (OR SKIP IF CSV EXISTS)
# ============================================================

NEWSAPI_KEY = "671a6b59db3148f3a2a4ae322aa18544"  # TODO: put your key here

LEFT_SOURCES   = ["the-guardian-uk", "cnn", "msnbc"]
RIGHT_SOURCES  = ["fox-news", "breitbart-news"]
CENTRE_SOURCES = ["reuters", "associated-press", "bbc-news"]

def fetch_headlines_from_sources(sources, label, page_size=100, pages=3):
    """
    Fetch headlines from a list of NewsAPI sources.
    page_size=100, pages=3 → up to 300 per source.
    """
    url = "https://newsapi.org/v2/top-headlines"
    rows = []
    for source in sources:
        for page in range(1, pages+1):
            params = {
                "apiKey": NEWSAPI_KEY,
                "sources": source,
                "pageSize": page_size,
                "page": page,
            }
            r = requests.get(url, params=params)
            if r.status_code != 200:
                print(f"[WARN] Source {source}, page {page}, status {r.status_code}, msg={r.text[:200]}")
                break
            data = r.json()
            articles = data.get("articles", [])
            if not articles:
                break
            for a in articles:
                title = a.get("title") or ""
                if not title.strip():
                    continue
                rows.append({"headline": title.strip(), "bias": label, "source": source})
        print(f"[INFO] Collected ~{len(rows)} rows so far for label '{label}'")
    return rows

# If you already have a train.csv from real data and don't want to call NewsAPI,
# you can comment out this whole block and just do:
# df = pd.read_csv("train.csv")

if NEWSAPI_KEY == "PUT_YOUR_NEWSAPI_KEY_HERE":
    print("⚠ Please set NEWSAPI_KEY with your actual key to fetch real news.")
    print("Skipping API fetch and exiting this cell.")
else:
    left_rows   = fetch_headlines_from_sources(LEFT_SOURCES,   "Left",   page_size=50, pages=5)
    right_rows  = fetch_headlines_from_sources(RIGHT_SOURCES,  "Right",  page_size=50, pages=5)
    centre_rows = fetch_headlines_from_sources(CENTRE_SOURCES, "Neutral",page_size=50, pages=5)

    all_rows = left_rows + right_rows + centre_rows
    df = pd.DataFrame(all_rows)
    df = df.drop_duplicates(subset=["headline"]).reset_index(drop=True)

    print("\nRaw collected distribution:")
    print(df["bias"].value_counts())

    # Optional: downsample or upsample to keep roughly balanced
    target_per_class_min = 800  # adjust to get 2k–5k total

    balanced_parts = []
    for label in ["Left","Right","Neutral"]:
        subset = df[df["bias"] == label]
        if len(subset) == 0:
            continue
        if len(subset) > target_per_class_min:
            subset = subset.sample(target_per_class_min, random_state=42)
        balanced_parts.append(subset)
    if balanced_parts:
        df = pd.concat(balanced_parts, axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

    print("\nBalanced-ish distribution:")
    print(df["bias"].value_counts())
    print("\nSample:")
    print(df.head())

    # Save dataset
    df.to_csv("train.csv", index=False)
    print("\n✅ Saved train.csv with", len(df), "rows.")


# ============================================================
# 2) LOAD train.csv (from NewsAPI or your own source)
# ============================================================
if not os.path.exists("train.csv"):
    raise SystemExit("❌ train.csv not found. Either set NEWSAPI_KEY or upload your own CSV to Colab.")

df = pd.read_csv("train.csv")
print("\nLoaded train.csv:")
print(df.head())
print("\nLabel distribution:")
print(df["bias"].value_counts())

# Ensure we have correct columns
if "headline" not in df.columns or "bias" not in df.columns:
    raise ValueError("CSV must contain 'headline' and 'bias' columns.")

df = df.rename(columns={"headline": "text", "bias": "label"})

# Filter out weird labels if any
VALID_LABELS = ["Left", "Right", "Neutral"]
df = df[df["label"].isin(VALID_LABELS)].reset_index(drop=True)

# Train/val split
train_df, val_df = train_test_split(df, test_size=0.1, stratify=df["label"], random_state=42)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))

# ============================================================
# 3) TOKENIZATION
# ============================================================
BASE = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE)

LABELS = ["Left","Neutral","Right"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

def tok(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_ds = train_ds.map(tok, batched=True)
val_ds   = val_ds.map(tok, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")

def encode(x):
    x["labels"] = label2id[x["labels"]]
    return x

train_ds = train_ds.map(encode)
val_ds   = val_ds.map(encode)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("\nTrain size:", len(train_ds), "Val size:", len(val_ds))


# ============================================================
# 4) MODEL + LoRA
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    BASE,
    num_labels=len(LABELS),
    label2id=label2id,
    id2label=id2label,
)

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    bias="none",
    task_type="SEQ_CLS",
)

peft_model = get_peft_model(model, lora_cfg)

# Make classifier head trainable
for name, p in peft_model.named_parameters():
    if "classifier" in name or "pre_classifier" in name:
        p.requires_grad = True

peft_model.print_trainable_parameters()

# ============================================================
# 5) TRAINING (20 EPOCHS)
# ============================================================
training_args = TrainingArguments(
    output_dir="./bias_model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=20,      # ← increased epochs
    weight_decay=0.01,
    learning_rate=4e-5,
    logging_steps=50,
    save_steps=1000,
    eval_steps=1000,
    report_to="none",
)

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
    }

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("\n🚀 Training starting...\n")
trainer.train()

print("\n✅ Training finished.")


# ============================================================
# 6) EXPORT FOR STREAMLIT
# ============================================================
SAVE_DIR = "/content/ft_lora_adapter"
os.makedirs(SAVE_DIR, exist_ok=True)

peft_model.save_pretrained(SAVE_DIR)

# Save classifier head
head_state = {
    "pre_classifier": peft_model.base_model.pre_classifier.state_dict(),
    "classifier": peft_model.base_model.classifier.state_dict(),
}
torch.save(head_state, os.path.join(SAVE_DIR, "cls_head.pt"))

# Save labels
with open("le_classes.txt", "w") as f:
    f.write(",".join(LABELS))

print("\n✅ Export complete.")
print("Saved: ft_lora_adapter/, cls_head.pt, le_classes.txt")

# Zip & download
!zip -r ft_lora_adapter.zip ft_lora_adapter
from google.colab import files
files.download("ft_lora_adapter.zip")
files.download("le_classes.txt")

print("\n🎉 ALL DONE! Upload to GitHub next to app.py")


[INFO] Collected ~0 rows so far for label 'Left'
[INFO] Collected ~10 rows so far for label 'Left'
[INFO] Collected ~20 rows so far for label 'Left'
[INFO] Collected ~10 rows so far for label 'Right'
[INFO] Collected ~20 rows so far for label 'Right'
[INFO] Collected ~0 rows so far for label 'Neutral'
[INFO] Collected ~10 rows so far for label 'Neutral'
[INFO] Collected ~20 rows so far for label 'Neutral'

Raw collected distribution:
bias
Left       20
Right      20
Neutral    20
Name: count, dtype: int64

Balanced-ish distribution:
bias
Left       20
Right      20
Neutral    20
Name: count, dtype: int64

Sample:
                                            headline     bias  \
0  Is the universe’s expansion slowing down? Astr...     Left   
1  Democrats are debating how to approach the new...     Left   
2  Economic Warning Signs Emerge. Here's What the...    Right   
3  NASA unveils close-up pictures of the interste...  Neutral   
4  Trump's absence from COP30 could be good for t...  

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 ALL DONE! Upload to GitHub next to app.py


In [3]:
# ============================================================
# FULL PIPELINE: Generate "realistic" bias dataset → Train LoRA → Export
# No external APIs, everything done inside Colab
# ============================================================

!pip install -q --upgrade transformers accelerate datasets peft scikit-learn

import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

# ============================================================
# 1) GENERATE MORE REALISTIC SYNTHETIC DATA (≈3000 ROWS)
# ============================================================
random.seed(42)

LEFT = "Left"
RIGHT = "Right"
NEUTRAL = "Neutral"

# Topics
policy_areas = [
    "healthcare reform",
    "climate policy",
    "tax legislation",
    "education funding",
    "labor protections",
    "immigration rules",
    "public housing program",
    "welfare expansion",
    "data privacy law",
    "minimum wage increase",
    "environmental regulations",
    "student loan relief",
    "policing reforms",
    "gun control measures",
    "corporate regulation",
    "social security changes",
    "LGBTQ+ rights protections",
]

countries = [
    "the United States",
    "India",
    "the United Kingdom",
    "Germany",
    "Canada",
    "Australia",
    "France",
    "Brazil",
]

actors_left = [
    "progressive lawmakers",
    "labor unions",
    "civil rights groups",
    "climate activists",
    "social justice advocates",
    "public sector workers",
    "teachers unions",
]

actors_right = [
    "business leaders",
    "conservative lawmakers",
    "right-leaning commentators",
    "industry lobbyists",
    "taxpayer associations",
    "security hawks",
]

actors_neutral = [
    "government officials",
    "central bank representatives",
    "policy analysts",
    "election commission officials",
    "independent watchdogs",
    "non-profit observers",
]

# Template pools
left_templates = [
    "{actor} in {country} push for {policy}, arguing it will reduce inequality.",
    "Editorial praises new {policy} as a major win for working families in {country}.",
    "Activists call {policy} a necessary step to protect vulnerable communities.",
    "Study suggests {policy} could close the wealth gap across {country}.",
    "Protesters demand urgent action on {policy} to tackle climate and social injustice.",
    "{actor} celebrate passage of {policy}, calling it a victory for ordinary citizens.",
    "New proposal on {policy} aims to expand public services and safety nets.",
    "Critics accuse corporations of blocking {policy} to protect their profits.",
]

right_templates = [
    "{actor} warn {policy} will hurt economic growth in {country}.",
    "Opinion piece slams {policy} as government overreach and an attack on freedom.",
    "Business groups argue {policy} will kill jobs and increase red tape.",
    "Commentators say {policy} punishes success and discourages investment.",
    "New plan on {policy} faces backlash for raising taxes and expanding bureaucracy.",
    "{actor} press for rolling back {policy}, citing risks to small businesses.",
    "Analysts claim {policy} threatens market competitiveness in {country}.",
    "Editorial criticizes {policy} for rewarding dependency and expanding welfare.",
]

neutral_templates = [
    "Government in {country} announces updated {policy} after months of debate.",
    "Officials outline timeline for implementing {policy} across key sectors.",
    "Experts give mixed reactions as {policy} moves through the legislature.",
    "Report details potential economic impact of {policy} in {country}.",
    "Parliament holds hearings to review details of {policy}.",
    "New data will be used to assess long-term effects of {policy}.",
    "Elections in {country} put future of {policy} in the spotlight.",
    "International observers track progress on {policy} implementation.",
]

prefixes = [
    "",
    "Analysis: ",
    "Opinion: ",
    "Report: ",
    "Breaking: ",
    "Update: ",
]

suffixes = [
    "",
    " amid growing public debate.",
    " raising questions about long-term effects.",
    " as elections approach.",
    " drawing attention from both supporters and critics.",
]

def generate_headlines(label, n):
    rows = []
    for _ in range(n):
        if label == LEFT:
            t = random.choice(left_templates)
            actor = random.choice(actors_left)
        elif label == RIGHT:
            t = random.choice(right_templates)
            actor = random.choice(actors_right)
        else:
            t = random.choice(neutral_templates)
            actor = random.choice(actors_neutral)

        policy = random.choice(policy_areas)
        country = random.choice(countries)
        prefix = random.choice(prefixes)
        suffix = random.choice(suffixes)

        headline = prefix + t.format(actor=actor, policy=policy, country=country) + suffix
        rows.append({"headline": headline, "bias": label})
    return rows

num_per_class = 1000  # 1000 Left, 1000 Right, 1000 Neutral → 3000 total

left_rows    = generate_headlines(LEFT, num_per_class)
right_rows   = generate_headlines(RIGHT, num_per_class)
neutral_rows = generate_headlines(NEUTRAL, num_per_class)

df = pd.DataFrame(left_rows + right_rows + neutral_rows).sample(frac=1, random_state=42).reset_index(drop=True)

print("Dataset distribution:")
print(df["bias"].value_counts())
print("\nSample rows:")
print(df.head())

df.to_csv("train.csv", index=False)
print("\n✅ Saved train.csv with", len(df), "rows.")


# ============================================================
# 2) PREPARE DATASET FOR TRAINING
# ============================================================
LABELS = [LEFT, NEUTRAL, RIGHT]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

df = df.rename(columns={"headline": "text", "bias": "label"})
train_df, val_df = train_test_split(df, test_size=0.1, stratify=df["label"], random_state=42)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))

print("\nTrain size:", len(train_ds), "Val size:", len(val_ds))


# ============================================================
# 3) TOKENIZATION
# ============================================================
BASE = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE)

def tok(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_ds = train_ds.map(tok, batched=True)
val_ds   = val_ds.map(tok, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")

def encode(x):
    x["labels"] = label2id[x["labels"]]
    return x

train_ds = train_ds.map(encode)
val_ds   = val_ds.map(encode)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


# ============================================================
# 4) MODEL + LoRA
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    BASE,
    num_labels=len(LABELS),
    label2id=label2id,
    id2label=id2label,
)

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    bias="none",
    task_type="SEQ_CLS",
)

peft_model = get_peft_model(model, lora_cfg)

# Make classifier head trainable
for name, p in peft_model.named_parameters():
    if "classifier" in name or "pre_classifier" in name:
        p.requires_grad = True

peft_model.print_trainable_parameters()


# ============================================================
# 5) TRAINING (20 EPOCHS)
#    - We keep arguments simple for compatibility (no evaluation_strategy)
# ============================================================
training_args = TrainingArguments(
    output_dir="./bias_model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=20,      #  ← increased epochs
    weight_decay=0.01,
    learning_rate=4e-5,
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    report_to="none",         # avoids wandb or other loggers
)

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
    }

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("\n🚀 Training starting...\n")
trainer.train()
print("\n✅ Training finished.")


# ============================================================
# 6) EXPORT FOR STREAMLIT
# ============================================================
SAVE_DIR = "/content/ft_lora_adapter"
os.makedirs(SAVE_DIR, exist_ok=True)

peft_model.save_pretrained(SAVE_DIR)

# Save classifier head separately
head_state = {
    "pre_classifier": peft_model.base_model.pre_classifier.state_dict(),
    "classifier":     peft_model.base_model.classifier.state_dict(),
}
torch.save(head_state, os.path.join(SAVE_DIR, "cls_head.pt"))

# Save labels
with open("le_classes.txt", "w") as f:
    f.write(",".join(LABELS))

print("\n✅ Export complete.")
print("Saved: ft_lora_adapter/, cls_head.pt, le_classes.txt")

# Zip & download for GitHub
!zip -r ft_lora_adapter.zip ft_lora_adapter
from google.colab import files
files.download("ft_lora_adapter.zip")
files.download("le_classes.txt")

print("\n🎉 ALL DONE! Upload these to your GitHub repo root next to app.py")


Dataset distribution:
bias
Right      1000
Left       1000
Neutral    1000
Name: count, dtype: int64

Sample rows:
                                            headline     bias
0  Report: Analysts claim welfare expansion threa...    Right
1  New plan on gun control measures faces backlas...    Right
2  Breaking: Opinion piece slams gun control meas...    Right
3  Report: Activists call student loan relief a n...     Left
4  Analysis: Elections in Canada put future of st...  Neutral

✅ Saved train.csv with 3000 rows.

Train size: 2700 Val size: 300


Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 ALL DONE! Upload these to your GitHub repo root next to app.py
